# Native PyTorch tutorial 5

Full mode generates one million rows, writes 100 original-format TFRecord shards and trains through two complete streamed passes. A native DataLoader batches each shard; global counters retain epoch/row information.

Run all cells in order. Full scientific workloads are the default; `NIF_SMOKE=1` is a separate development run and does not establish full convergence. The original TensorFlow and `_colab` notebooks are preserved as legacy references.

In [ ]:
# Run this cell first in a clean Colab Python 3.11–3.13 session.
import os, sys, subprocess, importlib.util
from pathlib import Path
# A local checkout is used directly for development; Colab installs the public pytorch branch.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "nif" / "model.py").exists()), None)
if repo is not None:
    sys.path.insert(0, str(repo))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.14.1", "numpy==2.1.3", "scipy==1.16.3", "matplotlib==3.10.0",
        "protobuf==7.36.2", "google-crc32c==1.9.0", "tfrecord==1.14.6",
        "git+https://github.com/pswpswpsw/nif.git@pytorch"])
import copy, json, time, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
import nif
# FULL is the default. Set NIF_SMOKE=1 before starting only for a short development check.
SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
device = torch.device(os.environ.get("NIF_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"torch": torch.__version__, "device": str(device), "mode": "SMOKE" if SMOKE else "FULL"})

OUT = Path(os.environ.get("NIF_OUTPUT_DIR", "pytorch_outputs")) / "tutorial5"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Explicit native loops replace compile/fit/callbacks. The loss reduction is per row,
# then the mean over the batch; area weights multiply each row's MSE (not weight-normalized).
def dense_tanh(inputs, outputs):
    # Preserve the original Dense default: Glorot uniform kernel and zero bias.
    layer = nn.Linear(inputs, outputs)
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return nn.Sequential(layer, nn.Tanh())

def epochs(full):
    return 2 if SMOKE else full

def tensor(array, on_device=device):
    return torch.as_tensor(array, dtype=torch.float32, device=on_device)

def loader(features, target, batch_size, weight=None):
    arrays = [tensor(features, "cpu"), tensor(target, "cpu")]
    if weight is not None:
        arrays.append(tensor(weight, "cpu").reshape(-1))
    return DataLoader(TensorDataset(*arrays), batch_size=batch_size, shuffle=True,
                      drop_last=False, pin_memory=device.type == "cuda")

def predict(model, features, batch_size=1024):
    model.eval()
    result = []
    with torch.no_grad():
        for xb in torch.as_tensor(features, dtype=torch.float32).split(batch_size):
            result.append(model(xb.to(next(model.parameters()).device)).detach().cpu())
    return torch.cat(result).numpy()

def centralize_gradients(model):
    # TF kernels are [input,output]; native Linear stores [output,input].
    # Centralize along the input axes, preserving the original mathematical operation.
    for module in model.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.grad is not None and parameter.ndim > 1:
                axes = tuple(range(1, parameter.ndim)) if isinstance(module, nn.Linear) else tuple(range(parameter.ndim-1))
                parameter.grad.sub_(parameter.grad.mean(dim=axes, keepdim=True))

def save_checkpoint(model, optimizer, path, epoch):
    payload = {"model_state": {k: v.detach().cpu() for k,v in model.state_dict().items()},
               "optimizer_state": optimizer.state_dict(), "epoch": epoch,
               "framework": "pytorch", "smoke": SMOKE}
    if hasattr(model, "cfg_shape_net"):
        payload.update(model_class=type(model).__name__, cfg_shape_net=model.cfg_shape_net,
                       cfg_parameter_net=model.cfg_parameter_net, mixed_policy="float32")
    torch.save(payload, path)

def save_and_check(model, optimizer, name, example, factory=None):
    path = OUT / (name + ".pt")
    save_checkpoint(model, optimizer, path, -1)
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if factory is None:
        factory = lambda: type(model)(copy.deepcopy(model.cfg_shape_net),
                                       copy.deepcopy(model.cfg_parameter_net), "float32")
    restored = factory().to(device)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval(); model.eval()
    with torch.no_grad():
        expected, actual = model(tensor(example)), restored(tensor(example))
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print("Checkpoint predictions verified:", path)
    return restored

def train(model, batches, optimizer, nepoch, name, schedule=None,
          centralization=False, loss_function=None, before_epoch=None):
    history = []
    start = time.monotonic()
    for epoch in range(nepoch):
        model.train()
        if schedule is not None:
            for group in optimizer.param_groups:
                group["lr"] = schedule(epoch, group["lr"])
        if before_epoch is not None:
            before_epoch(epoch)
        summed, seen = {}, 0
        for batch in batches:
            xb, yb = (item.to(device) for item in batch[:2])
            optimizer.zero_grad(set_to_none=True)
            if loss_function is None:
                per_row = (model(xb) - yb).square().mean(dim=-1)
                if len(batch) == 3:
                    per_row = per_row * batch[2].to(device).reshape(-1)
                data_loss = per_row.mean()
                values = {"mse": data_loss}
            else:
                data_loss, values = loss_function(model, xb, yb)
            regularization = model.regularization_loss(xb) if hasattr(model, "regularization_loss") else xb.new_zeros(())
            loss = data_loss + regularization
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {name}, epoch {epoch}")
            loss.backward()
            if centralization:
                centralize_gradients(model)
            optimizer.step()
            for key, value in {**values, "regularization": regularization, "loss": loss}.items():
                summed[key] = summed.get(key, 0.0) + float(value.detach()) * len(xb)
            seen += len(xb)
        row = {"epoch": epoch, "rows": seen, **{key: value / seen for key,value in summed.items()}}
        history.append(row)
        if epoch % 100 == 0 or epoch == nepoch - 1:
            print(name, row, "elapsed_seconds", round(time.monotonic()-start, 2))
            (OUT / (name + "_history.json")).write_text(json.dumps(history, indent=2))
        if epoch % 1000 == 0 or epoch == nepoch - 1:
            save_checkpoint(model, optimizer, OUT / (name + "_latest.pt"), epoch)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy([row["loss"] for row in history]); ax.set(xlabel="epoch", ylabel="loss", title=name)
    fig.tight_layout(); fig.savefig(OUT / (name + "_loss.png")); plt.show(); plt.close(fig)
    return history

def wave_figure(truth, prediction, name):
    fig, axs = plt.subplots(1,3,figsize=(15,3))
    for ax, values, title in zip(axs, (truth,prediction,prediction-truth), ("true","pred","error")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=50,cmap="seismic")
        fig.colorbar(im,ax=ax); ax.set(xlabel="t",ylabel="x",title=title)
    fig.tight_layout(); fig.savefig(OUT / (name+"_reconstruction.png")); plt.show(); plt.close(fig)

def write_metrics(values):
    print(json.dumps(values, indent=2))
    (OUT / "metrics.json").write_text(json.dumps({"mode": "smoke" if SMOKE else "full", **values}, indent=2))


# How to training on large dataset (>5GB) where data cannot fit in memory?

### Note 

- GPU memory is precious, don't waste it on dataset
- It can be super interesting to train NIF on HUGE dataset!
- In this Google Colab, I will show a demonstration using [tensorflow record](https://www.tensorflow.org/tutorials/load_data/tfrecord),
which is a simple format for storing a sequence of binary records.
- First you will learn how to obtain `tfrecord` file from `npz`, then you will
learn how to perform training across different `tfrecord` files.

## 0. Prepare a "large" numpy `npz` to play with

- this "ficitious" dataset has 7 dimensions, 
$$(t,x,y,z,u,v,w)$$
where $u,v,w$ are three components of velocity field. 
- Thus, we have **4 features, and 3 targets, no weight**. We will need this information to create the dataset 

- let's call it `Big_npz_file.npz`, it has 54 MB with $10^6$ points, each point contains 7 real number values
- it has a key argument `data`, which stores the data


In [ ]:
n_rows = 10000 if SMOKE else 1000000
points_per_shard = 1000 if SMOKE else 10000
data = np.random.uniform(0,1,(n_rows,7))
np.savez(OUT / "Big_npz_file.npz",data=data)


## 1. Generate `tfrecord` files from a big a `npz` 

First, you need to bring the `TFRDataset` class under `nif.data.tfr_dataset`

In [ ]:
from nif.data.tfr_dataset import TFRDataset


Second, create an instance, a "file handler", call it `fh`, given the `n_feature = 4` and `n_target = 3`

In [ ]:
fh = TFRDataset(n_feature=4,n_target=3)


### Then, you need to decide 
1. How many points to store in one single **tfrecord** file ----> `num_pts_per_file`
    - it doesn't have to be perfectly dividing the total number of points, we will make the last one smaller if it doesn't have a perfect divide.
    - Tensorflow official document [suggest](https://www.tensorflow.org/tutorials/load_data/tfrecord) around 100 MB.
    
2. What's the `npz` filepath? ----> `npz_path`
3. What's the key to access the 2-D matrix pointwise data? ----> `npz_key`
4. Where do you want to put the generated tfrecord file? ----> `tfr_path`
5. What's the prefix you want to put in the filename for the tfrecord file? 
    - they will be something like `prefix_0.npz`, ..., `prefix_10.npz`, ...

###  If you have answers to the above in your mind, now you can just call 
### `fh.create_from_npz(num_pts_per_file, npz_path, npz_key, write_tfr_path, prefix)`

In [ ]:
# Genuine TFRecord framing and tf.train.Example protobufs, without a TensorFlow runtime.
tfr_path = OUT / "TFR_dir"
fh.create_from_npz(num_pts_per_file=points_per_shard,npz_path=str(OUT / "Big_npz_file.npz"),
                   npz_key="data",tfr_path=str(tfr_path),prefix="case1")
shards=sorted(tfr_path.glob("*.tfrecord"))
assert len(shards) == (10 if SMOKE else 100)
print("TFRecord shards:",len(shards))


### Now, you can see the **tfrecord** files that you generated! 

# 2. How to train a PyTorch module on the tfrecord files that generated above?

### Let's create a toy Model first

In [ ]:
model=dense_tanh(4,3).to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,eps=1e-7)


Next, we generate a **meta dataset** or maybe you can call it **hyper dataset**. We will do something called **sub-dataset-batching** (I called it). 

- First you need to determine how many epoch you want to run ----> `epoch`
- Then we call the method `.get_tfr_meta_dataset(path, epoch)`

In [ ]:
epoch=2
meta_dataset=fh.get_tfr_meta_dataset(tfr_path=str(tfr_path),epoch=epoch)


### Finally, we do sub-dataset batching with `meta_dataset`

- we obtain `batch_file`, which are Tensors basically, from meta_dataset
- we create a `sub` dataset from `batch_file`

- you can add your favourite callbacks if you want
- the default `epoch` in the explicit training loop is 1, which makes sense in this context

In [ ]:
batch_size=128
rows_seen, shard_visits, accumulated_loss = 0,0,0.0
stream_losses=[]
# Two complete passes through every shard, including every final batch/remainder.
for batch_file in meta_dataset:
    batch_dataset=fh.gen_dataset_from_batch_file(batch_file,batch_size)
    shard_rows,shard_loss=0,0.0
    for xb,yb in batch_dataset:
        xb,yb=xb.to(device),yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss=(model(xb)-yb).square().mean()
        if not torch.isfinite(loss): raise FloatingPointError("non-finite streamed loss")
        loss.backward();optimizer.step()
        shard_rows+=len(xb);shard_loss+=float(loss.detach())*len(xb)
    rows_seen+=shard_rows;shard_visits+=1;accumulated_loss+=shard_loss
    stream_losses.append(shard_loss/shard_rows)
assert rows_seen == 2*n_rows
assert shard_visits == 2*len(shards)
model=save_and_check(model,optimizer,"streamed",data[:64,:4],factory=lambda:dense_tanh(4,3))
fig,ax=plt.subplots();ax.plot(stream_losses);ax.set(xlabel="shard visit",ylabel="MSE",title="Two streamed passes")
fig.savefig(OUT / "streamed_loss.png");plt.show();plt.close(fig)
write_metrics({"generated_rows":n_rows,"shards":len(shards),"passes":epoch,
               "rows_seen":rows_seen,"shard_visits":shard_visits,"mse":accumulated_loss/rows_seen})


### Cautious! 
- it is not perfect since we are `the training loop` for each `tfrecord` and this will make epoch information lost after each recall of `the training loop`
- we can hack it by using a training-loop hooks that explicitly write loss to the disk.. in an append way.
- **again what is presented here is not perfect, I am open for your solutions!**

### Question:

- **Why are you choosing to chop tfrecord like this?** It seems to me it is not what the official document has suggested, the syntax here looks very not software-optimized. 

### **Answer:** 

- Because the standard way of using `tfrecord` will take each data points, i.e., $(t,x,y,z,u,v,w)$ as an `tf.train.Example`. When you have 10 of millions, hundreds of millsion data points, **generating such dataset (not using this dataset for training)** has been reported to have performance issues if your algorithm contains any native Python Loop, which is not avoidable at the current stage. 

- In `nif.data.tfr_dataset.TFRDataset`, we subsampling npz file in a sequential order and take data along each dimension (e.g., (t_0,...,t_M) ) as an `tf.train.Example`, instead of treating each single point `(t,x,y,z,u,v,w)` as an example. Each shard has one `Example` containing seven column-vector features. 

- In this way, performance issue is greatly reduced. I believe the reason is that we leave the inner looping over all datapoints to the low-level

- So far, I can generate tfrecord from tens of Gigabyte data within 30 mins to 1 hour. While if I tried to make each training example as data point, it may take 

### Let me know <shawnpan@uw.edu>  if you have better solution for this, i.e., without using the meta-dataset to do sub-dataset-batching